In [1]:
import os
import re
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.metrics import classification_report, mean_absolute_error, r2_score
import joblib

In [7]:
# ==========================================
# CONFIG
# ==========================================
CONFIG = {
    "MAIN_TICKER": "000660.KS",  # ✅ SK hynix
    "TARGET_KIND": "regression",  # impact_score
    "N_SPLITS": 5,

    # 영향도 계산 가중치
    "WEIGHTS": {
        "return_weight": 0.4,     # 주가 반응
        "vol_weight": 0.2,        # 변동성
        "sentiment_weight": 0.2,  # 뉴스 감성 일치도
        "macro_weight": 0.2       # 환율/유가 등 반응
    }
}

In [8]:

# ==========================================
# 1️⃣ Load Data
# ==========================================
def read_csv_safe(path):
    if not os.path.exists(path):
        return None
    try:
        df = pd.read_csv(path)
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding="cp949")
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
    else:
        df["date"] = pd.date_range(end=datetime.today(), periods=len(df))
    return df

fred = read_csv_safe("../data/fred.csv")
market = read_csv_safe("../data/market_history_3y.csv")
yfq = read_csv_safe("../data/yf_quotes.csv")
ecos = read_csv_safe("../data/ecos_key_statistics.csv")

# ==========================================
# 2️⃣ Extract closing prices
# ==========================================
def extract_close(df):
    if df is None:
        return pd.DataFrame()

    out = pd.DataFrame({"date": df["date"]})
    for c in df.columns:
        if c.lower().startswith("date"):
            continue
        c_lower = str(c).lower()

        # 패턴 예시: KOSPI_('Close', '^KS11')
        m = re.search(r"_\('close'\s*,\s*'([^']+)'\)", c_lower)
        if m:
            sym = m.group(1).upper()
            out[f"{sym}__close"] = df[c]
    return out


px = extract_close(market).set_index("date").sort_index()

# ==========================================
# 3️⃣ Feature Engineering
# ==========================================
def build_features(px: pd.DataFrame, main_ticker: str):
    ret = px.pct_change()
    X = pd.DataFrame(index=px.index)

    main_col = f"{main_ticker}__close"
    if main_col in px.columns:
        X["pre_return_1d"] = np.log(px[main_col]).diff().shift(1)
        X["volatility_1d"] = X["pre_return_1d"].abs()
    else:
        X["pre_return_1d"] = X["volatility_1d"] = np.nan

    # 파생 변수 예시 (VIX, USDKRW, WTI 등)
    for sym in ["^VIX", "KRW=X", "CL=F"]:
        if f"{sym}__close" in px.columns:
            if "VIX" in sym:
                X["VIX_level"] = px[f"{sym}__close"].shift(1)
            elif "KRW" in sym:
                X["USD_KRW_chg_1d"] = px[f"{sym}__close"].pct_change().shift(1)
            elif "CL" in sym:
                X["WTI_chg_1d"] = px[f"{sym}__close"].pct_change().shift(1)

    return X.dropna(how="all")

X = build_features(px, CONFIG["MAIN_TICKER"])

# ==========================================
# 4️⃣ Target Definition (영향도 기반)
# ==========================================

bench = "^KS11"
if f"{bench}__close" in px.columns:
    ret_main = px[f"{CONFIG['MAIN_TICKER']}__close"].pct_change()
    ret_bench = px[f"{bench}__close"].pct_change()
    excess = ret_main - ret_bench
else:
    excess = px.pct_change().mean(axis=1)

# 1일 전 대비 변동성 변화
vol_change = X["volatility_1d"].diff()

# 매크로 반응 (환율, 유가)
macro_proxy = X[["USD_KRW_chg_1d", "WTI_chg_1d"]].abs().mean(axis=1)

# 뉴스·LLM alignment 점수 들어올 자리를 미리 만들어둠
# (나중에 fusion stage에서 join)
news_alignment = np.random.uniform(-1, 1, len(X))  # 예시 placeholder

# 영향도 점수 (가중 평균)
impact_score = (
    0.4 * np.abs(excess) +
    0.2 * vol_change.abs() +
    0.2 * np.abs(news_alignment) +
    0.2 * macro_proxy.fillna(0)
)

# 정규화
impact_score = (impact_score - impact_score.mean()) / impact_score.std()

# 분류 버전 라벨링
if CONFIG["TARGET_KIND"] == "classification":
    target = pd.qcut(impact_score, 3, labels=["low", "mid", "high"])
else:
    target = impact_score

dataset = X.join(pd.DataFrame({"target": target})).dropna()
dataset.to_csv(CONFIG["DATA_PATH"], index=True)


# ==========================================
# 5️⃣ Modeling & Hyperparameter Tuning
# ==========================================
tscv = TimeSeriesSplit(n_splits=CONFIG["N_SPLITS"])

if CONFIG["TARGET_KIND"] == "classification":
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("gbc", GradientBoostingClassifier(random_state=42))
    ])
    grid = {
        "gbc__n_estimators": [100, 200],
        "gbc__learning_rate": [0.05, 0.1, 0.2],
        "gbc__max_depth": [2, 3, 4]
    }
elif CONFIG["TARGET_KIND"] == "regression":
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("gbr", GradientBoostingRegressor(random_state=42))
    ])
    grid = {
        "gbr__n_estimators": [100, 200],
        "gbr__learning_rate": [0.05, 0.1, 0.2],
        "gbr__max_depth": [2, 3, 4]
    }

search = GridSearchCV(
    model,
    grid,
    cv=tscv,
    n_jobs=-1,
    scoring="accuracy" if CONFIG["TARGET_KIND"] == "classification" else "r2"
)

X_train = dataset.drop(columns="target")
y_train = dataset["target"]
search.fit(X_train, y_train)

best_model = search.best_estimator_
joblib.dump(best_model, CONFIG["MODEL_PATH"])

# ==========================================
# 6️⃣ Evaluate
# ==========================================
if CONFIG["TARGET_KIND"] == "classification":
    pred = best_model.predict(X_train)
    print(classification_report(y_train, pred))
else:
    pred = best_model.predict(X_train)
    print("MAE:", mean_absolute_error(y_train, pred))
    print("R2:", r2_score(y_train, pred))

print(f"✅ Model saved: {CONFIG['MODEL_PATH']}")
print(f"✅ Dataset saved: {CONFIG['DATA_PATH']}")


C:\Users\SKAX\AppData\Local\Temp\ipykernel_12352\923693825.py:49: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  ret = px.pct_change()
C:\Users\SKAX\AppData\Local\Temp\ipykernel_12352\923693825.py:67: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  X["WTI_chg_1d"] = px[f"{sym}__close"].pct_change().shift(1)
C:\Users\SKAX\AppData\Local\Temp\ipykernel_12352\923693825.py:79: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not 

KeyError: "['USD_KRW_chg_1d'] not in index"